# TP Integrador – Bases de Datos 2026

## Joaquín Erbity y Pedro Vignau

## Conexiones y limpieza

In [1]:
from pymongo import MongoClient
import redis
from neo4j import GraphDatabase

FECHA_REFERENCIA = "2026-10-01"

mongo = MongoClient("mongodb://localhost:27017")
db = mongo["tp_entradas"]

r = redis.Redis(host="localhost", port=6379, db=1, decode_responses=True)

neo = GraphDatabase.driver("bolt://127.0.0.1:7687", auth=("neo4j", "tpbases2026"))

# Prueba de conexión
print("Mongo:", mongo.server_info()["version"])
print("Redis:", r.ping())
neo.verify_connectivity()
print("Neo4j: conectado")

Mongo: 8.3.9
Redis: True
Neo4j: conectado


In [2]:
# Limpieza de las tres bases
mongo.drop_database("tp_entradas")
r.flushdb()
with neo.session() as s:
    s.run("MATCH (n) DETACH DELETE n")
print("Bases limpias")

Bases limpias


## 1. Carga inicial

In [3]:
usuarios = [
{"usuario_id": 1, "nombre": "Sofía Martínez", "email":
"sofia.martinez@example.com", "ciudad": "La Plata", "amigos": [2, 3], "sigue": [2,
5]},
{"usuario_id": 2, "nombre": "Juan López", "email": "juan.lopez@example.com",
"ciudad": "La Plata", "amigos": [1, 4, 5], "sigue": [1, 3]},
{"usuario_id": 3, "nombre": "Lucía Romero", "email": "lucia.romero@example.com",
"ciudad": "CABA", "amigos": [1, 6], "sigue": [2, 4]},
{"usuario_id": 4, "nombre": "Martín Díaz", "email": "martin.diaz@example.com",
"ciudad": "Rosario", "amigos": [2], "sigue": [3, 6]},{"usuario_id": 5, "nombre": "Valentina Ruiz", "email":
"valentina.ruiz@example.com", "ciudad": "Córdoba", "amigos": [2], "sigue": [1, 4,
6]},
{"usuario_id": 6, "nombre": "Tomás Herrera", "email":
"tomas.herrera@example.com", "ciudad": "La Plata", "amigos": [3], "sigue": [3, 6]}
]
artistas = [
{"artista_id": 1, "nombre": "Los Cóndores Eléctricos", "genero": "rock",
"similares": [6]},
{"artista_id": 2, "nombre": "La Marea Sur", "genero": "folk", "similares": [5]},
{"artista_id": 3, "nombre": "Pulso Nocturno", "genero": "electrónica",
"similares": [4]},
{"artista_id": 4, "nombre": "Cumbia Satelital", "genero": "cumbia", "similares":
[3]},
{"artista_id": 5, "nombre": "Viento Norte", "genero": "folk", "similares": [2]},
{"artista_id": 6, "nombre": "Radio Neón", "genero": "rock", "similares": [1]}
]
lugares = [
{"lugar_id": 1, "nombre": "Estadio Ciudad", "ciudad": "La Plata", "capacidad":
40000, "servicios": ["estacionamiento", "accesibilidad"]},
{"lugar_id": 2, "nombre": "Sala Diagonal", "ciudad": "La Plata", "capacidad":
800, "servicios": ["bar"]},
{"lugar_id": 3, "nombre": "Anfiteatro del Lago", "ciudad": "Córdoba",
"capacidad": 5000, "servicios": ["bar", "accesibilidad"]},
{"lugar_id": 4, "nombre": "Club Río", "ciudad": "Rosario", "capacidad": 2000,
"servicios": ["bar", "guardarropa"]}
]
eventos = [
{"evento_id": 1, "nombre": "Cóndores en vivo", "artistas": [1], "lugar_id": 1,
"fecha": "2026-05-10", "genero": "rock", "precio": 45000, "cupo": 500, "estado":
"activo"},
{"evento_id": 2, "nombre": "Marea acústica", "artistas": [2], "lugar_id": 2,
"fecha": "2026-06-20", "genero": "folk", "precio": 18000, "cupo": 150, "estado":
"activo"},
{"evento_id": 3, "nombre": "Noche Pulso", "artistas": [3], "lugar_id": 4,
"fecha": "2026-08-15", "genero": "electrónica", "precio": 25000, "cupo": 200,
"estado": "activo"},
{"evento_id": 4, "nombre": "Rock del Lago", "artistas": [1, 6], "lugar_id": 3,
"fecha": "2026-11-14", "genero": "rock", "precio": 38000, "cupo": 300, "estado":
"activo"},
{"evento_id": 5, "nombre": "Viento Norte en sala", "artistas": [5], "lugar_id":
2, "fecha": "2026-10-24", "genero": "folk", "precio": 20000, "cupo": 150, "estado":
"activo"},
{"evento_id": 6, "nombre": "Fiesta Satelital", "artistas": [4], "lugar_id": 1,
"fecha": "2026-12-05", "genero": "cumbia", "precio": 30000, "cupo": 500, "estado":
"activo"},
{"evento_id": 7, "nombre": "Radio Neón Tour", "artistas": [6], "lugar_id": 4,
"fecha": "2026-10-10", "genero": "rock", "precio": 22000, "cupo": 200, "estado":
"activo"},
{"evento_id": 8, "nombre": "Pulso Estadio", "artistas": [3], "lugar_id": 1,
"fecha": "2026-12-19", "genero": "electrónica", "precio": 42000, "cupo": 400,
"estado": "activo"} ]
compras = [
{"compra_id": 1, "usuario_id": 2, "evento_id": 1, "cantidad": 2,
"precio_unitario": 45000, "total": 90000, "fecha_compra": "2026-04-02", "estado":
"confirmada"},
{"compra_id": 2, "usuario_id": 5, "evento_id": 1, "cantidad": 1,
"precio_unitario": 45000, "total": 45000, "fecha_compra": "2026-04-10", "estado":
"confirmada"},
{"compra_id": 3, "usuario_id": 3, "evento_id": 1, "cantidad": 2,
"precio_unitario": 45000, "total": 90000, "fecha_compra": "2026-04-15", "estado":
"cancelada"},
{"compra_id": 4, "usuario_id": 1, "evento_id": 2, "cantidad": 2,
"precio_unitario": 18000, "total": 36000, "fecha_compra": "2026-06-01", "estado":
"confirmada"},
{"compra_id": 5, "usuario_id": 3, "evento_id": 2, "cantidad": 1,
"precio_unitario": 18000, "total": 18000, "fecha_compra": "2026-06-05", "estado":
"confirmada"},
{"compra_id": 6, "usuario_id": 4, "evento_id": 3, "cantidad": 3,
"precio_unitario": 25000, "total": 75000, "fecha_compra": "2026-07-20", "estado":
"confirmada"},
{"compra_id": 7, "usuario_id": 2, "evento_id": 3, "cantidad": 1,
"precio_unitario": 25000, "total": 25000, "fecha_compra": "2026-08-01", "estado":
"confirmada"},
{"compra_id": 8, "usuario_id": 1, "evento_id": 4, "cantidad": 2,
"precio_unitario": 38000, "total": 76000, "fecha_compra": "2026-09-10", "estado":
"confirmada"},
{"compra_id": 9, "usuario_id": 5, "evento_id": 6, "cantidad": 4,
"precio_unitario": 30000, "total": 120000, "fecha_compra": "2026-09-12", "estado":
"confirmada"},
{"compra_id": 10, "usuario_id": 6, "evento_id": 7, "cantidad": 1,
"precio_unitario": 22000, "total": 22000, "fecha_compra": "2026-09-15", "estado":
"confirmada"},
{"compra_id": 11, "usuario_id": 2, "evento_id": 8, "cantidad": 2,
"precio_unitario": 42000, "total": 84000, "fecha_compra": "2026-09-18", "estado":
"cancelada"},
{"compra_id": 12, "usuario_id": 4, "evento_id": 5, "cantidad": 2,
"precio_unitario": 20000, "total": 40000, "fecha_compra": "2026-09-20", "estado":
"confirmada"}
]
print(len(usuarios), len(artistas), len(lugares), len(eventos), len(compras))

6 6 4 8 12


In [4]:
# ---- Carga en MongoDB ----

# Usuarios: sin "amigos" ni "sigue" (eso va a Neo4j)
db.usuarios.insert_many([
    {k: v for k, v in u.items() if k not in ("amigos", "sigue")} for u in usuarios
])

# Artistas: sin "similares" (eso va a Neo4j)
db.artistas.insert_many([
    {k: v for k, v in a.items() if k != "similares"} for a in artistas
])

# Lugares, eventos y compras: completos
db.lugares.insert_many([dict(l) for l in lugares])
db.eventos.insert_many([dict(e) for e in eventos])
db.compras.insert_many([dict(c) for c in compras])

# Índices (para que no haya repetidos y las búsquedas sean rápidas)
db.usuarios.create_index("usuario_id", unique=True)
db.usuarios.create_index("email", unique=True)
db.artistas.create_index("artista_id", unique=True)
db.lugares.create_index("lugar_id", unique=True)
db.eventos.create_index("evento_id", unique=True)
db.compras.create_index("compra_id", unique=True)
db.compras.create_index("usuario_id")
db.compras.create_index("evento_id")

# Comprobación
for col in ["usuarios", "artistas", "lugares", "eventos", "compras"]:
    print(col, db[col].count_documents({}))

usuarios 6
artistas 6
lugares 4
eventos 8
compras 12


In [5]:
# ---- Carga en Neo4j ----
with neo.session() as s:
    # Restricciones de unicidad
    s.run("CREATE CONSTRAINT usuario_id IF NOT EXISTS "
          "FOR (u:Usuario) REQUIRE u.id IS UNIQUE")
    s.run("CREATE CONSTRAINT artista_id IF NOT EXISTS "
          "FOR (a:Artista) REQUIRE a.id IS UNIQUE")

    # Nodos: se crean leyendo desde MongoDB
    for u in db.usuarios.find():
        s.run("CREATE (:Usuario {id: $id, nombre: $nombre})",
              id=u["usuario_id"], nombre=u["nombre"])
    for a in db.artistas.find():
        s.run("CREATE (:Artista {id: $id, nombre: $nombre})",
              id=a["artista_id"], nombre=a["nombre"])

    # Relaciones: amistades y artistas seguidos
    for u in usuarios:
        for amigo in u["amigos"]:
            s.run("MATCH (a:Usuario {id: $a}), (b:Usuario {id: $b}) "
                  "MERGE (a)-[:AMIGO_DE]-(b)",
                  a=u["usuario_id"], b=amigo)
        for art in u["sigue"]:
            s.run("MATCH (u:Usuario {id: $u}), (a:Artista {id: $a}) "
                  "MERGE(u)-[:SIGUE]->(a)",
                  u=u["usuario_id"], a=art)

    # Relaciones: artistas similares
    for a in artistas:
        for sim in a["similares"]:
            s.run("MATCH (a:Artista {id: $a}), (b:Artista {id: $b}) "
                  "MERGE (a)-[:SIMILAR_A]-(b)",
                  a=a["artista_id"], b=sim)

    # Comprobación
    consultas = {
        "Usuarios":  "MATCH (u:Usuario) RETURN count(u) AS c",
        "Artistas":  "MATCH (a:Artista) RETURN count(a) AS c",
        "Amistades": "MATCH ()-[r:AMIGO_DE]->() RETURN count(r) AS c",
        "Sigue":     "MATCH ()-[r:SIGUE]->() RETURN count(r) AS c",
        "Similares": "MATCH ()-[r:SIMILAR_A]->() RETURN count(r) AS c",
    }
    for nombre, q in consultas.items():
        print(nombre + ":", s.run(q).single()["c"])

Usuarios: 6
Artistas: 6
Amistades: 5
Sigue: 13
Similares: 3


In [6]:
# ---- Carga en Redis ----

# Tiempos de vencimiento (en segundos)
TTL_SESION = 30 * 60     # 30 minutos
TTL_RESERVA = 10 * 60    # 10 minutos
TTL_BUSQUEDA = 5 * 60    # 5 minutos

# 1) Usuarios conectados: Sofía (1), Juan (2) y Tomás (6)
for uid in [1, 2, 6]:
    r.set(f"sesion:{uid}", "activa", ex=TTL_SESION)

# 2) Reservas sin confirmar
def crear_reserva(usuario_id, evento_id, cantidad):
    rid = r.incr("contador:reservas")
    clave = f"reserva:{rid}"
    r.hset(clave, mapping={
        "usuario_id": usuario_id,
        "evento_id": evento_id,
        "cantidad": cantidad,
    })
    r.expire(clave, TTL_RESERVA)
    return rid

crear_reserva(3, 5, 2)   # Lucía: 2 entradas para el evento 5
crear_reserva(6, 4, 3)   # Tomás: 3 entradas para el evento 4

# 3) Búsquedas por género (ranking)
r.zadd("busquedas:genero", {
    "rock": 7,
    "electrónica": 5,
    "folk": 3,
    "cumbia": 2,
})

# Comprobación
print("Sesiones:", sorted(r.keys("sesion:*")))
for clave in sorted(r.keys("reserva:*")):
    print(clave, r.hgetall(clave), "vence en", r.ttl(clave), "seg")
print("Búsquedas:", r.zrevrange("busquedas:genero", 0, -1, withscores=True))

Sesiones: ['sesion:1', 'sesion:2', 'sesion:6']
reserva:1 {'usuario_id': '3', 'evento_id': '5', 'cantidad': '2'} vence en 600 seg
reserva:2 {'usuario_id': '6', 'evento_id': '4', 'cantidad': '3'} vence en 600 seg
Búsquedas: [('rock', 7.0), ('electrónica', 5.0), ('folk', 3.0), ('cumbia', 2.0)]
